# Prueba Hibrido v01 — SpatialCLIP
## TSViT Spatial Encoder + CLIP Text Encoder

**Equipo 48** | Febrero 2026

**Arquitectura hibrida:** Un Image Encoder custom inspirado en el Spatial Encoder de
TSViT (Tarasiou et al., arXiv:2301.04944) con **K=18 CLS tokens aprendibles** (uno por
clase de cultivo), combinado con el Text Encoder pre-entrenado de CLIP (congelado) en
un framework contrastivo con perdida InfoNCE simetrica.

**Hipotesis:** Tener un CLS token dedicado por clase permite representaciones mas
discriminativas para imagenes satelitales multi-label, vs. el CLS token unico de un ViT estandar.

**Diferencia vs v1/v2:** El image encoder es 100% custom (no usa el ViT de CLIP).
Solo se reutiliza el text encoder de CLIP ViT-B/32 (congelado).

---
## 1. Imports y configuracion

In [1]:
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from PIL import Image
from torchvision import transforms as T

import open_clip
from sklearn.metrics import average_precision_score

plt.style.use("seaborn-v0_8-whitegrid")

# ── Reproducibilidad ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Dispositivo ──
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM: {vram_gb:.1f} GB")

# ── Hiperparametros ──
config = {
    "image_size": 128,       # PASTIS-R patches son 128x128
    "patch_size": 16,        # -> 8x8 = 64 patches
    "d_model": 256,          # dimension interna del spatial encoder
    "n_heads": 8,
    "n_layers": 6,           # capas del spatial encoder
    "k_classes": 18,         # numero de cls tokens (uno por clase)
    "d_clip": 512,           # dimension del espacio contrastivo (CLIP ViT-B/32)
    "batch_size": 64,
    "lr": 1e-4,
    "weight_decay": 0.01,
    "epochs": 50,
    "warmup_epochs": 5,
    "patience": 10,
}

N_PATCHES = (config["image_size"] // config["patch_size"]) ** 2  # 64
print(f"\nConfig: {json.dumps(config, indent=2)}")
print(f"Patches por imagen: {N_PATCHES}")

Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 4080 SUPER
  VRAM: 17.2 GB

Config: {
  "image_size": 128,
  "patch_size": 16,
  "d_model": 256,
  "n_heads": 8,
  "n_layers": 6,
  "k_classes": 18,
  "d_clip": 512,
  "batch_size": 64,
  "lr": 0.0001,
  "weight_decay": 0.01,
  "epochs": 50,
  "warmup_epochs": 5,
  "patience": 10
}
Patches por imagen: 64


---
## 2. Dataset y DataLoader

- Split oficial PASTIS: folds 1-4 train, fold 5 val (evita fuga geografica)
- Augmentations satelitales: rotacion 0/90/180/270, flip H/V, color jitter leve
- Caption augmentation: 3 templates aleatorios
- Los textos se retornan como strings crudos (se tokenizan en el forward del text encoder)

In [2]:
# ── Rutas y folds oficiales ──
DATA_DIR = Path("./train_data_v1")
METADATA_PATH = Path("../PASTIS-R/metadata.geojson")

with open(METADATA_PATH) as f:
    geojson = json.load(f)

patch_fold = {}
for feat in geojson["features"]:
    pid = int(feat["properties"]["ID_PATCH"])
    fold = int(feat["properties"]["Fold"])
    patch_fold[pid] = fold

available_ids = sorted([
    int(p.stem) for p in DATA_DIR.glob("*.png")
    if (DATA_DIR / f"{p.stem}.txt").exists()
])

train_ids = [pid for pid in available_ids if patch_fold.get(pid, 0) in [1, 2, 3, 4]]
val_ids   = [pid for pid in available_ids if patch_fold.get(pid, 0) == 5]

print(f"Patches disponibles: {len(available_ids):,}")
print(f"Train (folds 1-4):   {len(train_ids):,}")
print(f"Val   (fold 5):      {len(val_ids):,}")

# ── 18 clases oficiales de PASTIS ──
CROP_NAMES = {
    1: "meadow", 2: "soft winter wheat", 3: "corn", 4: "winter barley",
    5: "winter rapeseed", 6: "spring barley", 7: "sunflower", 8: "grapevine",
    9: "beet", 10: "winter triticale", 11: "winter durum wheat",
    12: "fruits vegetables and flowers", 13: "potatoes", 14: "leguminous fodder",
    15: "soybeans", 16: "orchard", 17: "mixed cereal", 18: "sorghum",
}
CROP_LIST = list(CROP_NAMES.values())
print(f"Clases: {len(CROP_LIST)}")

# ── Caption augmentation ──
CAPTION_TEMPLATES = [
    "A satellite image containing: {crops}",
    "satellite view of agricultural fields with {crops_and}",
    "aerial crop image: {crops}",
]


def augment_caption(original_caption: str) -> str:
    """Genera una variacion aleatoria del caption original."""
    prefix = "A satellite image containing: "
    crop_str = original_caption[len(prefix):] if original_caption.startswith(prefix) else original_caption
    crops = [c.strip() for c in crop_str.split(",")]
    template = random.choice(CAPTION_TEMPLATES)
    if "{crops_and}" in template:
        if len(crops) > 1:
            crops_and = ", ".join(crops[:-1]) + " and " + crops[-1]
        else:
            crops_and = crops[0]
        return template.format(crops_and=crops_and)
    return template.format(crops=", ".join(crops))


# ── Transforms ──
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

base_transform = T.Compose([
    T.Resize((config["image_size"], config["image_size"])),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

satellite_augment = T.Compose([
    T.RandomChoice([
        T.Lambda(lambda x: x),
        T.Lambda(lambda x: x.rotate(90, expand=False)),
        T.Lambda(lambda x: x.rotate(180, expand=False)),
        T.Lambda(lambda x: x.rotate(270, expand=False)),
    ]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.0),
])


# ── Dataset ──
class SatelliteHybridDataset(Dataset):
    """Dataset de pares (imagen, caption_string) para SpatialCLIP."""

    def __init__(self, data_dir, patch_ids, transform, augment=False):
        self.data_dir = Path(data_dir)
        self.patch_ids = patch_ids
        self.transform = transform
        self.augment = augment
        self.sat_augment = satellite_augment if augment else None

        # Pre-cargar captions en memoria
        self.captions = {}
        for pid in patch_ids:
            self.captions[pid] = (self.data_dir / f"{pid}.txt").read_text(encoding="utf-8").strip()

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, idx):
        pid = self.patch_ids[idx]
        img = Image.open(self.data_dir / f"{pid}.png").convert("RGB")

        if self.sat_augment is not None:
            img = self.sat_augment(img)

        img_tensor = self.transform(img)

        caption = self.captions[pid]
        if self.augment:
            caption = augment_caption(caption)

        return img_tensor, caption


def collate_fn(batch):
    """Collate: imagenes como tensor, textos como lista de strings."""
    images, captions = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, list(captions)


# ── DataLoaders ──
train_ds = SatelliteHybridDataset(DATA_DIR, train_ids, base_transform, augment=True)
val_ds   = SatelliteHybridDataset(DATA_DIR, val_ids,   base_transform, augment=False)

train_loader = DataLoader(
    train_ds, batch_size=config["batch_size"], shuffle=True,
    num_workers=2, pin_memory=(device == "cuda"), drop_last=True, collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_ds, batch_size=config["batch_size"], shuffle=False,
    num_workers=2, pin_memory=(device == "cuda"), collate_fn=collate_fn,
)

# Verificar un batch
images, captions = next(iter(train_loader))
print(f"\nBatch de imagenes: {images.shape}")
print(f"Batch de captions: {len(captions)} strings")
print(f"Ejemplo: '{captions[0][:80]}...'")
print(f"Batches/epoca:     {len(train_loader)}")

Patches disponibles: 2,433
Train (folds 1-4):   1,937
Val   (fold 5):      496
Clases: 18

Batch de imagenes: torch.Size([64, 3, 128, 128])
Batch de captions: 64 strings
Ejemplo: 'aerial crop image: grapevine, winter barley, winter durum wheat, meadow, legumin...'
Batches/epoca:     30


---
## 3. Modelo — SpatialEncoderCLIP

**Torre visual (custom):** Patch Embedding → K=18 CLS tokens + Spatial Positional Encodings → L=6 capas Transformer Encoder → Attention Pooling → Projection (256→512)

**Torre textual (frozen):** Text Encoder de CLIP ViT-B/32 pre-entrenado (congelado, sin gradientes)

```
Input imagen (B, 3, 128, 128)
  → PatchEmbedding: Conv2d → (B, 64, 256)
  → + Spatial Positional Encodings
  → Prepend K=18 CLS tokens → (B, 82, 256)
  → 6x TransformerEncoder (Pre-LN, 8 heads, GELU)
  → Extraer K CLS tokens → (B, 18, 256)
  → AttentionPooling (learnable query) → (B, 256)
  → Linear(256, 512) + L2 norm → (B, 512)
```

In [3]:
# ═══════════════════════════════════════════════════════════════
# (a) Patch Embedding
# ═══════════════════════════════════════════════════════════════

class PatchEmbedding(nn.Module):
    """Convierte imagen en secuencia de patch tokens via Conv2d."""

    def __init__(self, in_channels=3, d_model=256, patch_size=16):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, d_model,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, 3, H, W) -> (B, d_model, H/p, W/p)
        x = self.proj(x)
        # Flatten spatial -> (B, d_model, N) -> (B, N, d_model)
        x = x.flatten(2).transpose(1, 2)
        return x


# ═══════════════════════════════════════════════════════════════
# (b) Spatial Encoder (inspirado en TSViT Fig.4b)
# ═══════════════════════════════════════════════════════════════

class SpatialEncoder(nn.Module):
    """Transformer encoder con K CLS tokens aprendibles (uno por clase)."""

    def __init__(self, d_model=256, n_heads=8, n_layers=6,
                 n_patches=64, k_classes=18, dropout=0.1):
        super().__init__()
        self.k_classes = k_classes

        # K CLS tokens aprendibles (innovacion clave de TSViT)
        self.cls_tokens = nn.Parameter(torch.randn(1, k_classes, d_model) * 0.02)

        # Positional encodings para los patches espaciales
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.02)

        # Transformer Encoder (Pre-LN para estabilidad)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,  # Pre-LN (mas estable)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, patches):
        """
        patches: (B, N, d_model) — patch tokens del PatchEmbedding
        returns: (B, K, d_model) — K CLS token outputs
        """
        B = patches.shape[0]

        # Agregar positional encodings a los patches
        patches = patches + self.pos_embed

        # Prepend K CLS tokens
        cls = self.cls_tokens.expand(B, -1, -1)  # (B, K, d_model)
        x = torch.cat([cls, patches], dim=1)      # (B, K+N, d_model)

        # Transformer
        x = self.encoder(x)

        # Extraer solo los K CLS tokens
        cls_out = x[:, :self.k_classes, :]  # (B, K, d_model)
        cls_out = self.norm(cls_out)

        return cls_out


# ═══════════════════════════════════════════════════════════════
# (c) Attention Pooling
# ═══════════════════════════════════════════════════════════════

class AttentionPooling(nn.Module):
    """Learnable query que atiende a los K CLS tokens para producir un vector."""

    def __init__(self, d_model=256):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.scale = d_model ** -0.5

    def forward(self, cls_tokens):
        """
        cls_tokens: (B, K, d_model)
        returns: (B, d_model)
        """
        # q: (B, 1, d_model), K: (B, K, d_model)
        q = self.query.expand(cls_tokens.shape[0], -1, -1)

        # Attention: (B, 1, K)
        attn = (q @ cls_tokens.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        # Weighted sum: (B, 1, d_model) -> (B, d_model)
        out = (attn @ cls_tokens).squeeze(1)
        return out


# ═══════════════════════════════════════════════════════════════
# (d) Projection Head
# ═══════════════════════════════════════════════════════════════

class ProjectionHead(nn.Module):
    """Proyecta de d_model a d_clip con normalizacion L2."""

    def __init__(self, d_model=256, d_clip=512):
        super().__init__()
        self.proj = nn.Linear(d_model, d_clip)

    def forward(self, x):
        x = self.proj(x)
        x = F.normalize(x, dim=-1)
        return x


# ═══════════════════════════════════════════════════════════════
# (e) Frozen CLIP Text Encoder
# ═══════════════════════════════════════════════════════════════

class FrozenCLIPTextEncoder(nn.Module):
    """Wrapper del text encoder de CLIP (congelado, sin gradientes)."""

    def __init__(self, clip_model_name="ViT-B-32", pretrained="laion2b_s34b_b79k"):
        super().__init__()
        # Cargar modelo CLIP completo
        clip_model, _, _ = open_clip.create_model_and_transforms(
            clip_model_name, pretrained=pretrained
        )
        self.tokenizer = open_clip.get_tokenizer(clip_model_name)

        # Extraer componentes textuales
        self.token_embedding = clip_model.token_embedding
        self.positional_embedding = clip_model.positional_embedding
        self.transformer = clip_model.transformer
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.attn_mask = clip_model.attn_mask

        # Congelar todos los parametros
        for param in self.parameters():
            param.requires_grad = False

        # Eliminar visual encoder para liberar VRAM
        del clip_model.visual
        del clip_model
        torch.cuda.empty_cache()

    @torch.no_grad()
    def forward(self, texts):
        """
        texts: lista de strings
        returns: (B, d_clip) embeddings L2-normalizados
        """
        tokens = self.tokenizer(texts).to(next(self.parameters()).device)

        x = self.token_embedding(tokens)  # (B, seq_len, d_model)
        x = x + self.positional_embedding
        x = self.transformer(x, attn_mask=self.attn_mask)
        x = self.ln_final(x)

        # Tomar el token EOT (argmax de los token ids = posicion EOT)
        x = x[torch.arange(x.shape[0]), tokens.argmax(dim=-1)]

        if self.text_projection is not None:
            x = x @ self.text_projection

        x = F.normalize(x, dim=-1)
        return x


# ═══════════════════════════════════════════════════════════════
# (f) SpatialCLIP — modelo completo
# ═══════════════════════════════════════════════════════════════

class SpatialCLIP(nn.Module):
    """Modelo hibrido: Spatial Encoder (TSViT-inspired) + CLIP Text Encoder."""

    def __init__(self, config):
        super().__init__()

        # Torre visual (custom, entrenable)
        self.patch_embed = PatchEmbedding(
            in_channels=3,
            d_model=config["d_model"],
            patch_size=config["patch_size"],
        )
        self.spatial_encoder = SpatialEncoder(
            d_model=config["d_model"],
            n_heads=config["n_heads"],
            n_layers=config["n_layers"],
            n_patches=N_PATCHES,
            k_classes=config["k_classes"],
        )
        self.attn_pool = AttentionPooling(d_model=config["d_model"])
        self.projection = ProjectionHead(
            d_model=config["d_model"],
            d_clip=config["d_clip"],
        )

        # Torre textual (frozen CLIP)
        self.text_encoder = FrozenCLIPTextEncoder()

        # Temperatura aprendible (logit scale)
        self.logit_scale = nn.Parameter(torch.log(torch.tensor(1.0 / 0.07)))

    def encode_image(self, images):
        """images: (B, 3, H, W) -> (B, d_clip) L2-normalizado."""
        patches = self.patch_embed(images)        # (B, N, d_model)
        cls_out = self.spatial_encoder(patches)    # (B, K, d_model)
        pooled = self.attn_pool(cls_out)           # (B, d_model)
        emb = self.projection(pooled)              # (B, d_clip), L2-norm
        return emb

    def encode_text(self, texts):
        """texts: lista de strings -> (B, d_clip) L2-normalizado."""
        return self.text_encoder(texts)

    def forward(self, images, texts):
        image_embs = self.encode_image(images)
        text_embs = self.encode_text(texts)
        return image_embs, text_embs, self.logit_scale


# ── Instanciar y verificar ──
model = SpatialCLIP(config).to(device)

# Contar parametros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"SpatialCLIP creado:")
print(f"  Parametros totales:     {total_params:>12,}")
print(f"  Entrenables (visual):   {trainable_params:>12,}")
print(f"  Congelados (texto):     {frozen_params:>12,}")
print(f"  Tamano FP16:            {total_params * 2 / 1e6:.1f} MB")

# Smoke test
with torch.no_grad():
    dummy_imgs = torch.randn(4, 3, 128, 128, device=device)
    dummy_texts = ["A satellite image containing: meadow, corn"] * 4
    img_emb, txt_emb, ls = model(dummy_imgs, dummy_texts)
    print(f"\nSmoke test:")
    print(f"  Image embeddings: {img_emb.shape}")   # (4, 512)
    print(f"  Text embeddings:  {txt_emb.shape}")    # (4, 512)
    print(f"  Logit scale:      {ls.exp().item():.2f}")
    print(f"  Norma img[0]:     {img_emb[0].norm().item():.4f}")  # ~1.0
    print(f"  Norma txt[0]:     {txt_emb[0].norm().item():.4f}")  # ~1.0

/tmp/ipykernel_4123/2182035095.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

SpatialCLIP creado:
  Parametros totales:       68,516,865
  Entrenables (visual):      5,088,769
  Congelados (texto):       63,428,096
  Tamano FP16:            137.0 MB


RuntimeError: Expected all tensors to be on the same device, but got attn_bias is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA___scaled_dot_product_efficient_attention)

---
## 4. Funcion de perdida — InfoNCE simetrica

Perdida contrastiva estandar de CLIP: para un batch de N pares (imagen, texto), la diagonal de la matriz de similitud debe ser maxima. La temperatura es aprendible (logit_scale), clamped a max ln(100).

In [ ]:
def clip_loss(image_embs, text_embs, logit_scale):
    """
    InfoNCE simetrica.

    Args:
        image_embs: (N, d_clip) L2-normalizados
        text_embs:  (N, d_clip) L2-normalizados
        logit_scale: nn.Parameter (scalar)

    Returns:
        loss: scalar
    """
    # Temperatura aprendible, clamped para estabilidad
    scale = logit_scale.exp().clamp(max=100.0)

    # Matriz de similitud escalada (N, N)
    logits = scale * image_embs @ text_embs.t()

    # Labels: cada imagen i debe matchear con texto i
    labels = torch.arange(len(image_embs), device=image_embs.device)

    # Loss simetrica
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.t(), labels)
    loss = (loss_i2t + loss_t2i) / 2

    return loss


# Loss aleatoria esperada para batch=64: ln(64) ≈ 4.16
expected_random = math.log(config["batch_size"])
print(f"Loss function definida")
print(f"Loss aleatoria esperada (batch={config['batch_size']}): {expected_random:.2f}")

---
## 5. Training loop

- **Optimizer:** AdamW solo sobre parametros del image encoder + logit_scale (text encoder congelado)
- **Scheduler:** Cosine annealing con warmup lineal
- **Mixed precision:** FP16 autocast + GradScaler
- **Early stopping:** patience configurable sobre val_loss

In [ ]:
# ── Parametros entrenables (solo image encoder + logit_scale) ──
trainable_params_list = [
    p for n, p in model.named_parameters()
    if p.requires_grad
]
print(f"Parametros a optimizar: {sum(p.numel() for p in trainable_params_list):,}")

optimizer = torch.optim.AdamW(
    trainable_params_list,
    lr=config["lr"],
    weight_decay=config["weight_decay"],
)

# ── Scheduler: cosine annealing con warmup lineal ──
use_amp = device == "cuda"
scaler = GradScaler("cuda", enabled=use_amp)

warmup_steps = config["warmup_epochs"] * len(train_loader)
total_steps = config["epochs"] * len(train_loader)


def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"\nOptimizer:       AdamW (lr={config['lr']}, wd={config['weight_decay']})")
print(f"Scheduler:       Cosine + warmup ({config['warmup_epochs']} epochs)")
print(f"Steps/epoca:     {len(train_loader)}")
print(f"Total steps:     {total_steps}")
print(f"Warmup steps:    {warmup_steps}")
print(f"Mixed precision: {use_amp}")
print(f"Early stopping:  patience={config['patience']}")

# ═══════════════════════════════════════════════════════════════
# LOOP DE ENTRENAMIENTO
# ═══════════════════════════════════════════════════════════════

train_losses = []
val_losses = []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(config["epochs"]):
    # --- Train ---
    model.train()
    # Pero mantener text encoder en eval (batchnorm/dropout no aplican, pero por seguridad)
    model.text_encoder.eval()

    epoch_loss = 0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']} [train]")
    for images, captions in pbar:
        images = images.to(device)

        with autocast("cuda", enabled=use_amp):
            image_embs, text_embs, logit_scale = model(images, captions)
            loss = clip_loss(image_embs, text_embs, logit_scale)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params_list, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}",
            scale=f"{logit_scale.exp().item():.1f}",
        )

    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for images, captions in val_loader:
            images = images.to(device)
            with autocast("cuda", enabled=use_amp):
                image_embs, text_embs, logit_scale = model(images, captions)
                loss = clip_loss(image_embs, text_embs, logit_scale)
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches
    val_losses.append(avg_val_loss)

    # VRAM info
    vram_str = ""
    if device == "cuda":
        vram_used = torch.cuda.max_memory_allocated() / 1e9
        vram_str = f"  VRAM: {vram_used:.1f} GB"

    print(f"  Epoch {epoch+1}: train={avg_train_loss:.4f}  val={avg_val_loss:.4f}  "
          f"scale={logit_scale.exp().item():.1f}{vram_str}")

    # Checkpoint + early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "hibrido_v01_best.pt")
        print(f"  >>> Nuevo mejor modelo guardado (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= config["patience"]:
            print(f"  Early stopping en epoca {epoch+1} (sin mejora en {config['patience']} epocas)")
            break

print(f"\nEntrenamiento finalizado")
print(f"Epocas completadas: {len(train_losses)}")
print(f"Mejor val_loss:     {best_val_loss:.4f}")

# Guardar estado final
torch.save({
    "model_state_dict": model.state_dict(),
    "train_losses": train_losses,
    "val_losses": val_losses,
    "best_val_loss": best_val_loss,
    "config": config,
}, "hibrido_v01_final.pt")
print("Checkpoints guardados: hibrido_v01_best.pt, hibrido_v01_final.pt")

---
## 6. Evaluacion zero-shot

- Encode 18 prompts de cultivos con el text encoder
- Para cada imagen de val, calcular similitud coseno contra los 18 prompts
- **mAP** (mean Average Precision) como metrica principal para multi-label
- **Recall@K** para image-to-text retrieval
- **Per-crop F1** con threshold adaptativo

In [ ]:
# Cargar mejor checkpoint
model.load_state_dict(torch.load("hibrido_v01_best.pt", map_location=device, weights_only=True))
model.eval()
print(f"Mejor modelo cargado (val_loss={best_val_loss:.4f})")

# ── Encode 18 prompts de cultivos ──
crop_prompts = [f"A satellite image containing: {name}" for name in CROP_LIST]
with torch.no_grad():
    with autocast("cuda", enabled=use_amp):
        crop_text_feats = model.encode_text(crop_prompts)  # (18, 512)
        crop_text_feats = crop_text_feats.float().cpu()

print(f"Crop text features: {crop_text_feats.shape}")

# ── Encode todas las imagenes de validacion ──
all_img_feats = []
all_txt_feats = []

with torch.no_grad():
    for images, captions in tqdm(val_loader, desc="Encoding val set"):
        images = images.to(device)
        with autocast("cuda", enabled=use_amp):
            img_f = model.encode_image(images)
            txt_f = model.encode_text(captions)

        all_img_feats.append(img_f.float().cpu())
        all_txt_feats.append(txt_f.float().cpu())

all_img_feats = torch.cat(all_img_feats)  # (N_val, 512)
all_txt_feats = torch.cat(all_txt_feats)  # (N_val, 512)
print(f"Image features: {all_img_feats.shape}")

# ═══════════════════════════════════════════════════════════════
# Retrieval: Image-to-Text Recall@K
# ═══════════════════════════════════════════════════════════════

sim_matrix = all_img_feats @ all_txt_feats.t()  # (N, N)


def recall_at_k(sim, k):
    n = sim.shape[0]
    topk = sim.topk(k, dim=1).indices
    correct = torch.arange(n).unsqueeze(1).expand_as(topk)
    hits = (topk == correct).any(dim=1).float()
    return hits.mean().item()


r1 = recall_at_k(sim_matrix, 1)
r5 = recall_at_k(sim_matrix, 5)
r10 = recall_at_k(sim_matrix, 10)

print(f"\nImage-to-Text Retrieval (N={len(val_ids)}):")
print(f"  Recall@1:  {r1:.3f}")
print(f"  Recall@5:  {r5:.3f}")
print(f"  Recall@10: {r10:.3f}")

# ═══════════════════════════════════════════════════════════════
# Zero-shot classification: mAP y per-crop F1
# ═══════════════════════════════════════════════════════════════

# Scores de cada imagen vs 18 cultivos
crop_scores = all_img_feats @ crop_text_feats.t()  # (N_val, 18)

# Ground truth: vector binario (N_val, 18)
val_gt = np.zeros((len(val_ids), len(CROP_LIST)), dtype=np.float32)
for i, pid in enumerate(val_ids):
    caption = val_ds.captions[pid]
    prefix = "A satellite image containing: "
    crop_str = caption[len(prefix):] if caption.startswith(prefix) else caption
    crops_present = set(c.strip() for c in crop_str.split(","))
    for j, crop_name in enumerate(CROP_LIST):
        if crop_name in crops_present:
            val_gt[i, j] = 1.0

crop_scores_np = crop_scores.numpy()

# mAP (mean Average Precision) — metrica principal para multi-label
per_crop_ap = []
for j in range(len(CROP_LIST)):
    if val_gt[:, j].sum() > 0:  # al menos un positivo
        ap = average_precision_score(val_gt[:, j], crop_scores_np[:, j])
        per_crop_ap.append(ap)
    else:
        per_crop_ap.append(0.0)

mAP = np.mean(per_crop_ap)
print(f"\nZero-shot Classification:")
print(f"  mAP: {mAP:.4f}")

# Per-crop F1 con threshold adaptativo (mediana)
threshold = np.median(crop_scores_np)
print(f"  Threshold (mediana): {threshold:.4f}")

per_crop_results = []
for j, crop_name in enumerate(CROP_LIST):
    gt_pos = val_gt[:, j].astype(bool)
    n_pos = gt_pos.sum()
    n_neg = len(val_ids) - n_pos

    if n_pos == 0 or n_neg == 0:
        continue

    pred_pos = crop_scores_np[:, j] > threshold

    tp = (pred_pos & gt_pos).sum()
    fp = (pred_pos & ~gt_pos).sum()
    fn = (~pred_pos & gt_pos).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    per_crop_results.append({
        "crop": crop_name,
        "AP": per_crop_ap[j],
        "n_present": int(n_pos),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

df_crops = pd.DataFrame(per_crop_results).sort_values("f1", ascending=False)
print(f"\nPer-crop metrics:")
print(df_crops.to_string(index=False, float_format="{:.3f}".format))
print(f"\nMacro F1:  {df_crops['f1'].mean():.3f}")
print(f"mAP:       {mAP:.3f}")

---
## 7. Visualizacion de resultados

- Curvas de loss (train/val) + gap de overfitting
- Heatmap de similitud para un batch de validacion
- F1 y AP per-crop (bar charts)
- t-SNE de embeddings visuales coloreados por clase dominante
- Top-5 retrieval: dado un texto, mostrar las imagenes mas similares

In [ ]:
from sklearn.manifold import TSNE

# ═══════════════════════════════════════════════════════════════
# 7a. Curvas de loss + gap de overfitting
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

epochs_range = range(1, len(train_losses) + 1)

ax = axes[0]
ax.plot(epochs_range, train_losses, "o-", label="Train loss", color="#1976D2")
ax.plot(epochs_range, val_losses, "s-", label="Val loss", color="#F57C00")
best_epoch = val_losses.index(min(val_losses)) + 1
ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.5, label=f"Mejor epoca ({best_epoch})")
ax.set_xlabel("Epoca")
ax.set_ylabel("Loss (InfoNCE)")
ax.set_title("Curvas de entrenamiento — SpatialCLIP hibrido v01")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
gap = [t - v for t, v in zip(train_losses, val_losses)]
ax.plot(epochs_range, gap, "D-", color="#E53935", label="Train - Val (gap)")
ax.axhline(0, color="gray", linestyle="-", alpha=0.3)
ax.set_xlabel("Epoca")
ax.set_ylabel("Diferencia de loss")
ax.set_title("Gap de overfitting")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 7b. Heatmap de similitud (primeros 20 pares de validacion)
# ═══════════════════════════════════════════════════════════════

N_SHOW = min(20, len(val_ids))
sim_sub = sim_matrix[:N_SHOW, :N_SHOW].numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_sub, cmap="viridis", vmin=sim_sub.min(), vmax=sim_sub.max())
ax.set_xlabel("Texto (indice)")
ax.set_ylabel("Imagen (indice)")
ax.set_title(f"Matriz de similitud SpatialCLIP — primeros {N_SHOW} pares de val")
fig.colorbar(im, ax=ax, shrink=0.8)
for i in range(N_SHOW):
    ax.plot(i, i, "rx", markersize=8, markeredgewidth=2)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 7c. Per-crop F1 y AP (bar charts)
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# F1 por cultivo
ax = axes[0]
df_sorted = df_crops.sort_values("f1", ascending=True)
colors = ["#4CAF50" if f1 > 0.5 else "#FF9800" if f1 > 0.3 else "#F44336"
          for f1 in df_sorted["f1"]]
ax.barh(df_sorted["crop"], df_sorted["f1"], color=colors, edgecolor="none")
ax.set_xlabel("F1 Score")
ax.set_title("F1 per cultivo — SpatialCLIP hibrido v01")
ax.set_xlim(0, 1)
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.3)
ax.grid(True, alpha=0.2, axis="x")

# AP por cultivo
ax = axes[1]
df_ap = df_crops.sort_values("AP", ascending=True)
colors_ap = ["#4CAF50" if ap > 0.5 else "#FF9800" if ap > 0.3 else "#F44336"
             for ap in df_ap["AP"]]
ax.barh(df_ap["crop"], df_ap["AP"], color=colors_ap, edgecolor="none")
ax.set_xlabel("Average Precision")
ax.set_title(f"AP per cultivo (mAP={mAP:.3f})")
ax.set_xlim(0, 1)
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.3)
ax.grid(True, alpha=0.2, axis="x")

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 7d. t-SNE de embeddings visuales coloreados por clase dominante
# ═══════════════════════════════════════════════════════════════

# Clase dominante = primera clase mencionada en el caption (mayor area)
dominant_classes = []
for pid in val_ids:
    caption = val_ds.captions[pid]
    prefix = "A satellite image containing: "
    crop_str = caption[len(prefix):] if caption.startswith(prefix) else caption
    first_crop = crop_str.split(",")[0].strip()
    dominant_classes.append(first_crop)

# Solo las clases mas frecuentes para que el plot sea legible
from collections import Counter
class_counts = Counter(dominant_classes)
top_classes = [c for c, _ in class_counts.most_common(10)]

# Filtrar a las top-10 clases dominantes
mask = [dc in top_classes for dc in dominant_classes]
feats_filtered = all_img_feats[mask].numpy()
labels_filtered = [dc for dc, m in zip(dominant_classes, mask) if m]

print(f"t-SNE sobre {len(feats_filtered)} imagenes (top-10 clases dominantes)")

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, max_iter=1000)
coords = tsne.fit_transform(feats_filtered)

fig, ax = plt.subplots(figsize=(12, 10))
cmap = plt.colormaps["tab10"]
for i, cls_name in enumerate(top_classes):
    idx = [j for j, lbl in enumerate(labels_filtered) if lbl == cls_name]
    ax.scatter(coords[idx, 0], coords[idx, 1], c=[cmap(i)], label=cls_name,
               alpha=0.6, s=20, edgecolors="none")

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
ax.set_title("t-SNE de embeddings visuales — SpatialCLIP hibrido v01")
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 7e. Top-5 retrieval: texto -> imagenes mas similares
# ═══════════════════════════════════════════════════════════════

query_prompts = [
    "A satellite image containing: meadow",
    "A satellite image containing: corn, soft winter wheat",
    "A satellite image containing: grapevine",
    "A satellite image containing: sunflower",
]

for query in query_prompts:
    with torch.no_grad():
        with autocast("cuda", enabled=use_amp):
            q_feat = model.encode_text([query]).float().cpu()  # (1, 512)

    sims = (q_feat @ all_img_feats.t()).squeeze(0)  # (N_val,)
    top5_idx = sims.topk(5).indices.tolist()
    top5_scores = sims.topk(5).values.tolist()

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f'Query: "{query}"', fontsize=11)

    for rank, (idx, score) in enumerate(zip(top5_idx, top5_scores)):
        pid = val_ids[idx]
        img = Image.open(DATA_DIR / f"{pid}.png")
        axes[rank].imshow(img)
        axes[rank].axis("off")

        real_caption = val_ds.captions[pid].replace("A satellite image containing: ", "")
        title = f"#{rank+1} (sim={score:.3f})\n{real_caption[:50]}"
        axes[rank].set_title(title, fontsize=7)

    plt.tight_layout()
    plt.show()

print("\nVisualizacion completada.")